# MicroPy

A small, Turing-complete language with an executable reference interpreter.
Run the cells in order to inspect atomic evaluation steps and PENCIL context reduction. Examples cover object attributes, function calls, branching, bit copying/flipping, binary addition, and multiplication.


In [ ]:
import sys
sys.setrecursionlimit(max(sys.getrecursionlimit(), 10000))

from typing import List, Tuple

def list_str(l:List):
    if len(l) == 0:
        return ""
    return str(l[0]) + suffix_str(l[1:])

def suffix_str(l:List):
    if len(l) == 0:
        return ""
    return ' ' + str(l[0]) + suffix_str(l[1:])

def list_str_tokens(l: List):
    if len(l) == 0:
        return "empty"
    return list_str(l)

class Effect(): pass


In [ ]:
def clear_state():
    global base_state
    global bit_index
    global max_length
    max_length = 0
    bit_index = 0
    base_state = Effect([])

def clear_funs():
    global base_funs
    global exp_index
    global exp_defs
    base_funs = []
    exp_index = 0
    exp_defs = []

def print_response(s:str):

    global next_prompt
    global max_length
    max_length = max(max_length,ntokens(s))

    print(f"{s[:len(next_prompt)]} [EOP] {s[len(next_prompt)+1:]}")

    if s[len(s)-5:] != "[ret]":
        print()
        print("response does not end in [ret]")
        raise Exception("Response does not end in [ret]")

    if s[:len(next_prompt)] != next_prompt:
        print()
        print("PENCIL Error")
        raise Exception(f"PENCIL Error")

    arrow_index = s.rfind(" => ")
    call_index = s[:arrow_index].rfind(" [call] ")
    response = s[arrow_index+4:len(s)-6]
    next_prompt = f"{s[:call_index]} {response}"

def ntokens(s:str):
    if len(s) == 0: return 0
    i = s.rfind(" ")
    if i == -1: return 1
    if i == len(s)-1: return ntokens(s[:len(s)-1])
    return 1 + ntokens(s[:i])


In [ ]:
class Val_Or_TailApply(): pass

class Value(Val_Or_TailApply): pass

class Obj(Value):
    def __init__(self,name:str):
        self.name = name
    def __str__(self):
        return f"O . {self.name}"

class Bool(Value):
    def __init__(self,name:bool):
        self.name = name
    def __str__(self):
        return f"B . {self.name}"

class Unit(Value):

    def __init__(self):
        pass
    def __str__(self):
        return f"U . Unit"

class Failure(Value):

    def __init__(self):
        pass
    def __str__(self):
        return f"Fa . Failure"

true_ = Bool(True)
false_ = Bool(False)
unit_ = Unit()
fail_ = Failure()

class Attr():
    def __init__(self,name:str):
        self.name = name
    def __str__(self):
        return f"Att . {self.name}"

class Assertion():
    def __init__(self, obj:Obj, attr:Attr, val:Value):
        self.obj = obj
        self.attr = attr
        self.val = val
    def __str__(self):
        return f"Assertion( {self.obj} {self.attr} {self.val} )"

class Effect():
    def __init__(self,assertions:List[Assertion]):
        self.assertions = assertions
    def __str__(self):
        return f"Eff( {list_str_tokens(self.assertions)} )"

def add_effect(ef1:Effect,ef2:Effect) -> Effect:
    return Effect(ef1.assertions + ef2.assertions)

def lookup_attr(obj:Obj,attr:Attr,s:Effect) -> Value:
    for assertion in reversed(s.assertions):
        if obj.name == assertion.obj.name and attr.name == assertion.attr.name:
            return assertion.val
    raise Exception(f"no value fournd for {obj}.{attr}")

def MPhasattr(obj:Obj,attr:Attr,s:Effect) -> bool:
    for assertion in s.assertions:
        if obj.name == assertion.obj.name and attr.name == assertion.attr.name:
            return true_
    return false_

class Binding():

    def __init__(self,name:str,val:Value):
        self.name = name
        self.val = val
    def __str__(self):
        return f"Bind( {self.name} {self.val} )"

class Env():
    def __init__(self,bindings:List[Binding]):
        self.bindings = bindings
    def __str__(self):
        return f"Env( {list_str(self.bindings)} )"

def lookup_var(name:str,env:Env) -> Value:
    for binding in env.bindings:
        if binding.name == name:
            return binding.val
    raise Exception(f"no value found for variable {name}")

class Exp(): pass

def set_name(e:Exp):
    global exp_index
    global exp_defs
    exp_index += 1
    name = f"Exp{exp_index}"
    exp_defs = exp_defs + [f"D . {name} = {e}"]
    e.name = name

class TailApply():

    def __init__(self,f,args: List[Value]):
        self.f = f
        self.args = args
    def __str__(self):
        return f"TailApp( {self.f.name} Args( {list_str(self.args)} ) )"


In [ ]:
def name(e):
    return e.name if isinstance(e,Exp) else e

def run_program(e:Exp):
    global fun_defs
    global exp_defs
    global base_state
    def MicroPy_def(fun):
        return f"FD {fun.name} = lambda( {list_str(fun.varnames)} ) {name(fun.body())} ;"
    fundefs = list_str([MicroPy_def(f) for f in base_funs])
    base_stack = f"FunDefs( {fundefs} ) ********* ExpDefs( {list_str(exp_defs)} ) ****** {base_state} ********* Env( )"
    global next_prompt
    next_prompt = (f"{base_stack} [call] [call] {name(e)}")
    ef,val = eval(e,base_stack,base_state,Env([]))
    base_state = add_effect(base_state,ef)
    print(f"{ef} {val}      max context = {max_length} tokens")

def eval(e, stack: str, s: Effect, env:Env)->Tuple[Effect,Value]:

    if isinstance(e,Value):
        print_response (f"{stack} [call] {e} => {Effect([])} {e} [ret]")
        return Effect([]),e

    ef,cval = tailval(e, f"{stack} [call]", s, env)

    while isinstance(cval,TailApply):
        print_response(f"{stack} [call] {ef} {cval} => [call] {ef} [call] {cval} [ret]")
        ef2, next_cval = compute_TailApply(cval,f"{stack} [call] {ef}",add_effect(s,ef))
        next_ef = add_effect(ef,ef2)
        print_response(f"{stack} [call] {ef} {ef2} {next_cval} => [call] {next_ef} {next_cval} [ret]")
        cval = next_cval
        ef = next_ef
    print_response(f"{stack} [call] {ef} {cval} => {ef} {cval} [ret]")
    return ef, cval

def tailval(e, stack: str, s: Effect, env:Env) -> Tuple[Effect,Val_Or_TailApply]:

    if isinstance(e,Value):
        print_response (f"{stack} [call] {e} => {Effect([])} {e} [ret]")
        return Effect([]),e

    print_response(f"{stack} [call] {name(e)} => [call] {e} [ret]")
    return e.tailval(stack,s,env)

def compute_TailApply(app:TailApply,stack:str,s:Effect) -> Tuple[Effect,Val_Or_TailApply]:

    body = app.f.body()
    print_response(f"{stack} [call] {app} => [call] let( {list_str_tokens(app.f.varnames)} ) be ( {list_str_tokens(app.args)} ) in {name(body)} [ret]")
    env = Env([Binding(name,val) for name,val in zip(app.f.varnames,app.args)])
    print_response(f"{stack} [call] let( {list_str_tokens(app.f.varnames)} ) be ( {list_str_tokens(app.args)} ) in {name(body)} => [call] {env} [call] {name(body)} [ret]")
    ef,val = tailval(body,f"{stack} [call] {env}",s,env)
    print_response(f"{stack} [call] {env} {ef} {val} => {ef} {val} [ret]")
    return ef,val


In [ ]:
class Equal(Exp):

    def __init__(self,exp1:Exp,exp2:Exp):
        self.exp1 = exp1
        self.exp2 = exp2
        set_name(self)
    def __str__(self):
        return f"Equal( {name(self.exp1)} {name(self.exp2)} )"

    def tailval(self,stack:str,s:Effect,env:Env) -> Tuple[Effect,Bool]:

        ef1,val1 = eval(self.exp1,f"{stack} [call] {self} exp1=",s,env)
        ef2,val2 = eval(self.exp2,f"{stack} [call] {self} exp1= {ef1} {val1} exp2=",add_effect(s,ef1),env)
        combined_ef = add_effect(ef1,ef2)
        rval = true_ if str(val1) == str(val2) else false_
        print_response(f"{stack} [call] {self} exp1= {ef1} {val1} exp2= {ef2} {val2} => {combined_ef} {rval} [ret]")
        return combined_ef,rval


In [ ]:
clear_funs()
clear_state()
run_program(Equal(Obj("A"),Obj("B")))

print()
clear_funs()
clear_state()
run_program(Equal(Obj("A"),Obj("A")))


In [ ]:
class App(Exp):

    def __init__(self,f,args:List[Exp]):
        self.f = f
        self.args = args
        set_name(self)

    def __str__(self):
        return f"App( {self.f.name} ( {list_str([name(e) for e in self.args])} ))"

    def tailval(self, stack:str, s:Effect, env:Env) -> Tuple[Effect,Val_Or_TailApply]:
        argvals = []
        args_effect = Effect([])
        args_state=s
        args_stack = f"{stack} [call] {self}"
        for arg,i in zip(self.args,range(len(self.args))):
            ef,val = eval(arg,f"{args_stack} arg{i}=",args_state,env)
            argvals = argvals + [val]
            args_effect = add_effect(args_effect,ef)
            args_state = add_effect(args_state,ef)
            args_stack = f"{args_stack} arg{i}= {ef} {val}"

        app = TailApply(self.f,argvals)
        print_response(f"{args_stack} => {args_effect} {app} [ret]")
        return args_effect, app


In [ ]:
class LookupVar(Exp):

    def __init__(self, var:str):
        self.var = var;
        set_name(self)

    def __str__(self):
        return f"LookupVar( {self.var} )"

    def tailval(self, stack:str, s:Effect,env:Env) -> Tuple[Effect,Value]:
        val = lookup_var(self.var,env)
        ef = Effect([])
        print_response(f"{stack} [call] {self} => {ef} {val} [ret]")
        return ef,val


In [ ]:
class MicroPyFun():
    def __init__(self,func):
        self.func = func
        self.name = func.__name__
        self.varnames = func.__code__.co_varnames
    def body(self):
        if hasattr(self,"body_cache"): return self.body_cache
        self.body_cache = self.func(*[LookupVar(name) for name in self.varnames])
        return self.body_cache
    def __str__(self):
        return f"F.{self.name}"

def make_symbolic(fun):
    return lambda *args: App(fun,args)

def MicroPy(func):
    global base_funs
    f = MicroPyFun(func)
    base_funs += [f]
    return make_symbolic(f)


In [ ]:
clear_funs()
clear_state()

@MicroPy
def f(x,y):
    return Equal(x,y)

run_program(f(Obj("A"),Obj("B")))


In [ ]:
class Seq(Exp):

    def __init__(self,x:Exp,y:Exp):
        self.x = x
        self.y = y
        set_name(self)

    def __str__(self):
        return f"Seq( {name(self.x)} , {name(self.y)} )"

    def tailval(self, stack:str, s:Effect, env:Env) -> Tuple[Effect,Val_Or_TailApply]:
        xef,xval = eval(self.x,f"{stack} [call] {self} x=",s,env)
        print_response(f"{stack} [call] {self} x= {xef} {xval} => [call] {xef} [call] {name(self.y)} [ret]")
        yef,yval = tailval(self.y,f"{stack} [call] {xef}",add_effect(s,xef),env)
        combined_ef = add_effect(xef,yef)
        print_response(f"{stack} [call] {xef} {yef} {yval} => {combined_ef} {yval} [ret]")
        return combined_ef,yval


In [ ]:
clear_funs()
clear_state()

run_program(Seq(Obj("A"),Obj("B")))


In [ ]:
class If(Exp):

    def __init__(self, test:Exp, then:Exp, els:Exp):
        self.test = test
        self.then = then
        self.els = els
        set_name(self)

    def __str__(self):
        return f"If( {name(self.test)} {name(self.then)} {name(self.els)} )"

    def tailval(self,stack:str,s:Effect,env:Env) -> Tuple[Effect,Val_Or_TailApply]:
        testef,testval = eval(self.test,f"{stack} [call] {self} test=",s,env)
        case = self.then if testval == true_ else self.els
        print_response(f"{stack} [call] {self} test= {testef} {testval} => [call] {testef} [call] {name(case)} [ret]")
        caseef,caseval = tailval(case,f"{stack} [call] {testef}",add_effect(s,testef),env)
        combined_ef = add_effect(testef,caseef)
        print_response(f"{stack} [call] {testef} {caseef} {caseval} => {combined_ef} {caseval} [ret]")
        return combined_ef,caseval


In [ ]:
clear_funs()
clear_state()

run_program(If(true_,Obj("a"),Obj("b")))
print("")

clear_funs()
clear_state()

run_program(If(false_,Obj("a"),Obj("b")))
print ("")


In [ ]:
class Assert(Exp):
    def __init__(self, objexp:Exp, attr:Attr, valexp:Exp):
        self.objexp = objexp
        self.attr = attr
        self.valexp = valexp
        set_name(self)

    def __str__(self):
        return f"Assert( {name(self.objexp)} {self.attr} {name(self.valexp)} )"

    def tailval(self,stack,s:Effect,env:Env) -> Tuple[Effect,Unit]:
        ef1,obj = eval(self.objexp,f"{stack} [call] {self} obj=",s, env)
        ef2,val = eval(self.valexp,f"{stack} [call] {self} obj= {ef1} {obj} val=", add_effect(s,ef1), env)
        total_effect = add_effect(ef1, add_effect(ef2, Effect([Assertion(obj,self.attr,val)])))
        print_response(f"{stack} [call] {self} obj= {ef1} {obj} val= {ef2} {val} => {total_effect} {unit_} [ret]")
        return total_effect,unit_


In [ ]:
clear_funs()
clear_state()

run_program(Assert(Obj("A"), Attr("B"), Obj("C")))


In [ ]:
class HasAttr(Exp):
    def __init__(self, objexp:Exp, attr:Attr):
        self.objexp = objexp
        self.attr = attr
        set_name(self)

    def __str__(self):
        return f"HasAttr( {name(self.objexp)} {self.attr} )"

    def tailval(self,stack,s:Effect,env:Env) -> Tuple[Effect,Value]:
        ef,obj = eval(self.objexp,f"{stack} [call] {self} obj=",s, env)
        val = MPhasattr(obj,self.attr,add_effect(s,ef))
        print_response(f"{stack} [call] {self} obj= {ef} {obj} => {ef} {val} [ret]")
        return ef,val


In [ ]:
clear_funs()
clear_state()

run_program(Assert(Obj("A"), Attr("B"), Obj("C")))
print()

run_program(HasAttr(Obj("A"),Attr("B")))
print()

run_program(HasAttr(Obj("A"),Attr("D")))


In [ ]:
class LookupAttr(Exp):
    def __init__(self, objexp:Exp, attr:Attr):
        self.objexp = objexp
        self.attr = attr
        set_name(self)

    def __str__(self):
        return f"LookupAttr( {name(self.objexp)} {self.attr} )"

    def tailval(self,stack,s:Effect,env:Env) -> Tuple[Effect,Value]:
        ef,obj = eval(self.objexp,f"{stack} [call] {self} obj=",s, env)
        val = lookup_attr(obj,self.attr,add_effect(s,ef))
        print_response(f"{stack} [call] {self} obj= {ef} {obj} => {ef} {val} [ret]")
        return ef,val


In [ ]:
clear_funs()
clear_state()

run_program(Assert(Obj("A"), Attr("B"), Obj("C")))
print()
run_program(LookupAttr(Obj("A"),Attr("B")))


In [ ]:
class Try(Exp):
    def __init__(self,first_attempt:'Exp',second_attempt:'Exp'):
        self.first_attempt = first_attempt
        self.second_attempt = second_attempt
        set_name(self)

    def __str__(self):
        return f"Try( {name(self.first_attempt)} {name(self.second_attempt)} )"

    def tailval(self,stack:str,s:Effect,env:Env) -> Tuple[Effect,Val_Or_TailApply]:
        ef,v = eval(self.first_attempt,f"{stack} [call] {self} [call] first=",s,env)
        if v != fail_:
            print_response(f"{stack} [call] {self} [call] first= {ef} {v} => {ef} {v} [ret]")
            print_response(f"{stack} [call] {self} {ef} {v} => {ef} {v} [ret]")
            return ef,v
        print_response(f"{stack} [call] {self} [call] first= {ef} {v} => second= [ret]")
        ef,v = eval(self.second_attempt,f"{stack} [call] {self} second=",s,env)
        if v != fail_:
            print_response(f"{stack} [call] {self} second= {ef} {v} => {ef} {v} [ret]")
            return ef,v
        noef = Effect([])
        print_response(f"{stack} [call] {self} second= {ef} {v} => {noef} {fail_} [ret]")
        return noef,fail_


In [ ]:
clear_funs()
clear_state()

run_program(Try(Seq(Assert(Obj("a"),Attr("b"),Obj("c")),Obj("success")),Seq(Assert(Obj("a"),Attr("b"),Obj("d")),Obj("success"))))
print("")

clear_state()
run_program(Try(Seq(Assert(Obj("a"),Attr("b"),Obj("c")),fail_),Seq(Assert(Obj("a"),Attr("b"),Obj("d")),Obj("success"))))


In [ ]:
def new_bit()->Obj:
    global bit_index
    bit_index += 1
    return Obj(f"bit{bit_index}")

def add_assertion(obj:Obj,attr:Attr,val:Value):
    global base_state
    base_state = add_effect(base_state,Effect([Assertion(obj,attr,val)]))

def new_bit_string(n:int)->Obj:
    if n==1:
        return new_bit()
    brest = new_bit_string(n-1)
    bnext = new_bit()
    add_assertion(bnext,Attr("next"),brest)
    return bnext

def randomize_bit_string(b:Obj):
    add_assertion(b,Attr("value"),random.choice([true_,false_]))
    if MPhasattr(b,Attr("next"),base_state) == true_:
        randomize_bit_string(lookup_attr(b,Attr("next"),base_state))

def bit_string_values(b:Obj)->List[Obj]:
    v = lookup_attr(b,Attr("value"),base_state)
    if MPhasattr(b,Attr("next"),base_state) == true_:
        return [v] + bit_string_values(lookup_attr(b,Attr("next"),base_state))
    return [v]

def bit_string_value(b:Obj)->int:
    if not MPhasattr(b,Attr("value"),base_state) == true_:
        return 0
    v = lookup_attr(b,Attr("value"),base_state) == true_
    if MPhasattr(b,Attr("next"),base_state) == true_:
        return 2*bit_string_value(lookup_attr(b,Attr("next"),base_state)) + (1 if v else 0)
    return 1 if v else 0

def reverse_index(b:Obj):
    if MPhasattr(b,Attr("next"),base_state) == true_:
        next = lookup_attr(b,Attr("next"),base_state)
        reverse_index(next)
        add_assertion(next,Attr("prev"),b)

def bit_string_list(b:Obj)->List[Obj]:
    if MPhasattr(b,Attr("next"),base_state) == true_:
        return [b] + bit_string_list(lookup_attr(b,Attr("next"),base_state))
    return [b]

def zero_bit_string(b:Obj):
    add_assertion(b,Attr("value"),false_)
    if MPhasattr(b,Attr("next"),base_state) == true_:
        zero_bit_string(lookup_attr(b,Attr("next"),base_state))

def set_bit_string(b,n:int):
    if n % 2 == 0:
        add_assertion(b,Attr("value"),false_)
    else:
        add_assertion(b,Attr("value"),true_)
    if MPhasattr(b,Attr("next"),base_state) == true_:
        return set_bit_string(lookup_attr(b,Attr("next"),base_state),n//2)

def nth_bit(b:Obj,n:int)->Obj:
    if n == 0:
        return b
    return nth_bit(lookup_attr(b,Attr("next"),base_state),n-1)


In [ ]:
import random
clear_state()
clear_funs()

copy_len = 10

b1 = new_bit_string(copy_len)
b2 = new_bit_string(copy_len)

randomize_bit_string(b1)

@MicroPy
def copy_bits(b1,b2):
    return Seq(Assert(b2,Attr("value"),LookupAttr(b1,Attr("value"))),
                If(HasAttr(b1,Attr("next")),
                    copy_bits(LookupAttr(b1,Attr("next")),
                              LookupAttr(b2,Attr("next"))),
                    unit_))

run_program(copy_bits(b1,b2))
print(list_str(bit_string_values(b1)))
print(list_str(bit_string_values(b2)))
print(f"max context = {max_length} tokens")


In [ ]:
clear_state()
clear_funs()

flip_len = 10
b1 = new_bit_string(flip_len)
b2 = new_bit_string(flip_len)
randomize_bit_string(b1)

@MicroPy
def not_(p):
    return If(p,false_,true_)

@MicroPy
def flip_bits(b1,b2):
    return Seq(Assert(b2,Attr("value"),not_(LookupAttr(b1,Attr("value")))),
                If(HasAttr(b1,Attr("next")),
                    flip_bits(LookupAttr(b1,Attr("next")),
                                LookupAttr(b2,Attr("next"))),
                    unit_))

run_program(flip_bits(b1,b2))

print("")
print("")
print(list_str(bit_string_values(b1)))
print(list_str(bit_string_values(b2)))

print("")
print(f"max context = {max_length} bytestokens")


In [ ]:
import random
clear_funs()

@MicroPy
def exor_(p,q):
    return If(p,If(q,false_,true_),If(q,true_,false_))

@MicroPy
def and_(p,q):
    return If(p,q,false_)

@MicroPy
def or_(p,q):
    return If(p,true_,q)

@MicroPy
def rpc_sum_(a1,a2,c):
    return exor_(a1,exor_(a2,c))

@MicroPy
def rpc_carry_(a1,a2,c):
    return or_(and_(a1,a2),and_(c,or_(a1,a2)))

@MicroPy
def RPC_add(b1,b2,b3,c):
    return RPC_aux(b1,b2,b3,c,rpc_carry_(LookupAttr(b1,Attr("value")),
                                         LookupAttr(b2,Attr("value")),
                                         c))

@MicroPy
def RPC_aux(b1,b2,b3,c,nc):
    return Seq(Assert(b3,Attr("value"),rpc_sum_(LookupAttr(b1,Attr("value")),LookupAttr(b2,Attr("value")),c)),
                If(HasAttr(b1,Attr("next")),
                    RPC_add(LookupAttr(b1,Attr("next")),
                            LookupAttr(b2,Attr("next")),
                            LookupAttr(b3,Attr("next")),
                            nc),
                    Assert(LookupAttr(b3,Attr("next")),Attr("value"),nc)))

clear_state()

rpc_len = 3

b1 = new_bit_string(rpc_len)
randomize_bit_string(b1)
b2 = new_bit_string(rpc_len)
randomize_bit_string(b2)
print(f"b1 = {bit_string_value(b1)}")
print(f"b2 = {bit_string_value(b2)}")
b3 = new_bit_string(rpc_len+1)
run_program(RPC_add(b1,b2,b3,false_))
print(f"b1+ b2 = {bit_string_value(b3)}")

print(f"max context = {max_length} tokens")


In [ ]:
clear_state()

rpc_len = 3

b1 = new_bit_string(rpc_len)
randomize_bit_string(b1)
b2 = new_bit_string(rpc_len)
randomize_bit_string(b2)
print(f"b1 = {bit_string_value(b1)}")
print(f"b2 = {bit_string_value(b2)}")
b3 = new_bit_string(rpc_len+1)
zero_bit_string(b3)
run_program(RPC_add(b1,b2,b3,false_))
print(f"b1+ b2 = {bit_string_value(b3)}")

print(f"max context = {max_length} tokens")


In [ ]:
@MicroPy
def RPC_mult(b1,b2,b3):
    return Seq(If(LookupAttr(b1,Attr("value")),
                    RPC_add(b2,b3,b3,false_),
                    unit_),
                If(HasAttr(b1,Attr("next")),
                    RPC_mult(LookupAttr(b1,Attr("next")),b2,LookupAttr(b3,Attr("next"))),
                    unit_))

clear_state()
mul_len = 3
b1 = new_bit_string(mul_len)
randomize_bit_string(b1)
b2 = new_bit_string(mul_len)
randomize_bit_string(b2)
b3 = new_bit_string(2*mul_len+4)
zero_bit_string(b3)

print(f"bit string b1 = {list_str(bit_string_list(b1))}")
print(f"bit string b2 = {list_str(bit_string_list(b2))}")
print(f"bit string b3 = {list_str(bit_string_list(b3))}")

print(f"b1 = {bit_string_value(b1)}")
print(f"b2 = {bit_string_value(b2)}")
run_program(RPC_mult(b1,b2,b3))
print(list_str(bit_string_values(b1)))
print(list_str(bit_string_values(b2)))
print(list_str(bit_string_values(b3)))
print(f"b1 * b2 = {bit_string_value(b3)}")
print(f"max context = {max_length} tokens")
